In [10]:
%pip install -q openai python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [13]:
import json
import os
from openai import OpenAI
import ast
import pandas as pd
from dotenv import load_dotenv

load_dotenv("../.env")

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

MODELS_CONFIG = {
    "gemini": "google/gemini-2.5-flash",
    "gemma": "google/gemma-3-27b-it",
    "qwen": "qwen/qwen3-vl-32b-instruct"
}

ACTIVE_MODEL = "gemini"
current_model_id = MODELS_CONFIG[ACTIVE_MODEL]
print(f"Đang sử dụng model: [{ACTIVE_MODEL.upper()}] - {current_model_id}")

Đang sử dụng model: [GEMINI] - google/gemini-2.5-flash


In [6]:
BASE_DIR = "../"  

FILE_TOP3_URLS = BASE_DIR + "database/retrieval_eval_outputs/top3_urls_long.csv"
FILE_REFINED   = BASE_DIR + "refined/refined_outputs_openrouter/refined_gemini-2.5-flash.csv"
FILE_CORPUS    = BASE_DIR + "chunking_scripts/final_corpus.csv"

COL_CLAIM_ID = "id"
COL_TOP3_TEXT = "top3_text_urls" # Cột chứa mảng dạng chuỗi: "['url1', 'url2', 'url3']"
COL_REFINED_JSON = "raw_output" # Cột chứa chuỗi JSON của Refined Claim
COL_CORPUS_URL = "url"
COL_CORPUS_TEXT = "content" # Cột chứa nội dung bài báo/bằng chứng

# # (Tùy chọn) Chọn Experiment tốt nhất để chạy (vì file long.csv có nhiều experiment)
# BEST_EXPERIMENT_ID = "gemini-2.5-flash_semantic_clip_finetuned_True"

print("Đang nạp dữ liệu từ các file CSV...")

df_corpus = pd.read_csv(FILE_CORPUS)
df_corpus = df_corpus.drop_duplicates(subset=[COL_CORPUS_URL])
corpus_dict = df_corpus.set_index(COL_CORPUS_URL)[COL_CORPUS_TEXT].to_dict()

df_refined = pd.read_csv(FILE_REFINED)
df_refined['refined_dict'] = df_refined[COL_REFINED_JSON].apply(lambda x: json.loads(x) if pd.notnull(x) else {})

df_top3 = pd.read_csv(FILE_TOP3_URLS)
# # Lọc theo Experiment tốt nhất (nếu bạn chạy file _long.csv)
# if 'experiment_id' in df_top3.columns:
#     df_top3 = df_top3[df_top3['experiment_id'] == BEST_EXPERIMENT_ID]

# Parse chuỗi "['url1', 'url2']" thành list Python
df_top3[COL_TOP3_TEXT] = df_top3[COL_TOP3_TEXT].apply(lambda x: ast.literal_eval(x) if pd.notnull(x) else [])

# 3.4 Merge Top3 URLs và Refined Claims dựa trên Claim ID
df_merged = pd.merge(df_top3, df_refined, on=COL_CLAIM_ID, how='inner')
print(f"Đã chuẩn bị xong {len(df_merged)} claims để Fact-checking!")

Đang nạp dữ liệu từ các file CSV...
Đã chuẩn bị xong 160 claims để Fact-checking!


In [ ]:
# HÀM ĐÁNH GIÁ TỪNG BẰNG CHỨNG DỰA TRÊN REFINED CLAIM
def evaluate_single_evidence(refined_claim, evidence_text, evidence_index):
    # Trích xuất dữ liệu đã được tiền xử lý
    norm_claim = refined_claim.get("normalized_claim", "")

    # Lấy danh sách các mệnh đề cần kiểm tra
    # atoms_json = refined_claim.get("claim_atoms", "").apply(lambda x: json.loads(x) if pd.notnull(x) else {})
    atoms = [f"+ {atom['text']} (Ưu tiên: {atom['priority']})" for atom in refined_claim.get("claim_atoms", [])]
    atoms_str = "\n".join(atoms) if atoms else "Không có mệnh đề cụ thể."

    # Lấy các quan sát từ ảnh (THAY THẾ HOÀN TOÀN VIỆC ĐỌC ẢNH)
    visuals = [f"+ {v['text']}" for v in refined_claim.get("visual_observations", [])]
    visuals_str = "\n".join(visuals) if visuals else "Không có thông tin thị giác."

    system_prompt = f"""You are a Fact-Checking expert.
    Task: Compare a structurally analyzed Claim with a SINGLE piece of Evidence.

    [RULES]: Only use information from this Evidence. DO NOT use external knowledge.
    [NOTE]: YOU MUST RETURN ONLY A VALID JSON FORMAT. DO NOT WRAP IN MARKDOWN BLOCKS.

    Required JSON format:
    {{
      "thought_process": "Evaluate whether this Evidence addresses any of the 'Claim Atoms' or 'Visual Observations'.",
      "relation": "SUPPORT | REFUTE | PARTIAL_SUPPORT | UNRELATED",
      "extracted_facts": "Extract concise sentences/events from the Evidence that are DIRECTLY RELATED to the Atoms (max 3 bullet points). If the Evidence is unrelated, leave as an empty string ''."
    }}

    Relation label notes:
    - SUPPORT: Fully supports the Claim.
    - REFUTE: Directly contradicts the Claim.
    - PARTIAL_SUPPORT: Partially matches, but lacks information to confirm the entire Claim.
    - UNRELATED: Not related to the Claim.
    """

    user_prompt = f"""[CLAIM STRUCTURE TO VERIFY]:
    - Claim: {norm_claim}
    - Claim Atoms to consider: {atoms_str}
    - Visual observations from the attached image: {visuals_str}

    -------------------
    [EVIDENCE {evidence_index}]: {evidence_text}
    """

    user_content = [{"type": "text", "text": user_prompt}]

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_content}
    ]

    try:
        response = client.chat.completions.create(
            model=current_model_id,
            messages=messages,
            temperature=0.0,
            response_format={"type": "json_object"}
        )
        return json.loads(response.choices[0].message.content)
    except Exception as e:
        print(f"Lỗi khi xử lý Evidence {evidence_index}: {e}")
        return None


def get_final_verdict(refined_claim, map_results):
    norm_claim = refined_claim.get("normalized_claim", "")

    # Lấy checklist mục tiêu xác minh
    targets = [f"- {t}" for t in refined_claim.get("verification_targets", [])]
    targets_str = "\n".join(targets) if targets else "Không có mục tiêu cụ thể."

    system_prompt = """You are a Fact-Checking expert.
    Based on the Claim, the list of Verification Targets, and the Extracted Information Pieces from the evidence, provide the final verdict.

    MANDATORY RULES:
    - If the synthesized pieces provide enough evidence to confirm the entire Claim -> SUPPORTED.
    - If any piece directly contradicts the Claim -> REFUTED.
    - If the pieces lack sufficient information or are unrelated -> NEI (Not Enough Information).
    NOTE: RETURN ONLY A VALID JSON FORMAT. DO NOT WRAP IN MARKDOWN BLOCKS.

    REQUIREMENTS FOR EXPLANATION:
    - Write naturally and directly explain why the Claim is true, false, or lacks evidence.
    - Must provide specific facts, numbers, or quotes as evidence if any.
    - Written entirely in Vietnamese.
    - ABSOLUTELY DO NOT mention the technical process (do not use phrases like "synthesizing evidence", "combining information", "piece number 1", "source 1", etc.). State it as a self-evident truth.

    Reason and return in the following JSON format:
    {
      "thought_process": {
        "synthesis": "Synthesize the extracted information pieces from all Sources.",
        "target_check": "Check if the information pieces address the 'Verification Targets'. If there are no Targets, compare the synthesized information against the Claim. Is there enough evidence? Are there contradictions?",
        "logical_deduction": "Logical conclusion."
      },
      "verdict": "SUPPORTED | REFUTED | NEI",
      "explanation": "A complete explanation for the end-user fully in VIETNAMESE (under 50 words), containing specific evidence if any."
    }"""

    # Format lại các facts từ bước Map để đưa vào prompt
    compiled_facts = ""
    for idx, res in enumerate(map_results):
        if res and res.get('relation') != 'UNRELATED':
            compiled_facts += f"\n- Nguồn {idx + 1} ({res.get('relation')}): {res.get('extracted_facts')}"

    if not compiled_facts.strip():
        compiled_facts = "\nNo relevant information regarding the Claim from the provided sources."

    user_prompt = f"""[CLAIM]: {norm_claim}

    [VERIFICATION TARGETS (CHECKLIST)]: {targets_str}

    [EXTRACTED INFORMATION PIECES FROM EVIDENCE]: {compiled_facts}
    """

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    try:
        response = client.chat.completions.create(
            model=current_model_id,
            messages=messages,
            temperature=0.0,
            response_format={"type": "json_object"}
        )
        return json.loads(response.choices[0].message.content)
    except Exception as e:
        print(f"Lỗi Phán quyết cuối: {e}")
        return None

In [ ]:
results_list = []
max_claims_to_process = 10  # Giới hạn số claim để test, bạn có thể tăng lên sau khi kiểm tra ổn định
df_merged = df_merged.head(max_claims_to_process)

print(f"\nBắt đầu Fact-Checking {len(df_merged)} claims...")
for index, row in df_merged.iterrows():
    claim_id = row[COL_CLAIM_ID]
    refined_claim = row['refined_dict']
    top_urls = row[COL_TOP3_TEXT]

    # print(f"\n{'='*60}\n Đang xử lý Claim ID: {claim_id}\n Tuyên bố: {refined_claim.get('normalized_claim', '')}\n{'='*60}")

    # Bóc tách text cho top 3 URLs từ corpus
    evidences_text = []
    for url in top_urls:
        if url in corpus_dict:
            evidences_text.append(str(corpus_dict[url]))
        else:
            print(f" [Cảnh báo] Không tìm thấy URL trong Corpus: {url}")

    # Chạy Map (Đánh giá từng evidence)
    map_results = []
    for idx, ev_text in enumerate(evidences_text):
        result = evaluate_single_evidence(refined_claim, ev_text, idx + 1)
        map_results.append(result)

    # Chạy Reduce (Tổng hợp)
    final_verdict = get_final_verdict(refined_claim, map_results)

    # print(f"VERDICT: {final_verdict.get('verdict')}")
    # print(f"EXPLANATION: {final_verdict.get('explanation')}")

    # Lưu kết quả
    results_list.append({
        "claim_id": claim_id,
        "normalized_claim": refined_claim.get('normalized_claim'),
        "verdict": final_verdict.get("verdict"),
        "explanation": final_verdict.get("explanation"),
        "thought_process": json.dumps(final_verdict.get("thought_process", {}), ensure_ascii=False),
        "top3_urls_used": top_urls,
        "map_results": json.dumps(map_results, ensure_ascii=False)
    })

# =====================================================================
# 6. XUẤT KẾT QUẢ RA FILE
# =====================================================================
output_file = f"factchecking_final_results_{ACTIVE_MODEL}.csv"
df_results = pd.DataFrame(results_list)
df_results.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"\nĐã lưu kết quả thành công tại: {output_file}")


Bắt đầu Fact-Checking 10 claims...

 Đang xử lý Claim ID: 4
 Tuyên bố: Tổng số tiền bị chiếm đoạt bởi đường dây lừa đảo này ước tính là 100 tỷ đồng.
VERDICT: SUPPORTED
EXPLANATION: Tổng số tiền bị chiếm đoạt bởi các đường dây lừa đảo được ghi nhận là hơn 100 tỷ đồng. Cụ thể, một nhóm đã chiếm đoạt 1.300 tỷ đồng (Nguồn 1), một nhóm khác hơn 160 tỷ đồng (Nguồn 3), và các nguồn khác cũng báo cáo các khoản tiền lớn bị lừa đảo.

 Đang xử lý Claim ID: 6
 Tuyên bố: Đường dây lừa đảo này chỉ nhắm mục tiêu vào người bị hại tại tỉnh Thái Bình.
VERDICT: REFUTED
EXPLANATION: Đường dây lừa đảo này hoạt động trên toàn quốc, với các vụ việc được ghi nhận tại nhiều tỉnh thành như Đà Nẵng, Hà Nội, TP.HCM, và Cao Bằng, chứ không chỉ giới hạn tại Thái Bình. Tổng số tiền lừa đảo lên đến hơn 160 tỷ đồng từ gần 400 công dân trên cả nước.

 Đang xử lý Claim ID: 5
 Tuyên bố: Các đối tượng lừa đảo đã bị bắt giữ vào ngày 15 tháng 5 năm 2024.
VERDICT: REFUTED
EXPLANATION: Thông tin về thời gian bắt giữ các đối 